In [ ]:
import re
import nltk
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

try:
    STOP_WORDS = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords")
    STOP_WORDS = set(stopwords.words("english"))

stemmer = PorterStemmer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    words = text.split()
    words = [w for w in words if w not in STOP_WORDS]
    words = [stemmer.stem(w) for w in words]

    return words

if __name__ == "__main__":
    text1 = "The Artificial Intelligence is changing the world!"
    text2 = "Artificial intelligence changes our world."

    print("Text 1:", preprocess_text(text1))
    print("Text 2:", preprocess_text(text2))

Text 1: ['artifici', 'intellig', 'chang', 'world']
Text 2: ['artifici', 'intellig', 'chang', 'world']


In [ ]:

import random

def initialize_population(size, length):
    population = []
    for i in range(size):
        individual = [random.randint(0, 1) for j in range(length)]
        population.append(individual)
    return population

def fitness_function(individual, text1, text2):
    words = sorted(set(text1) | set(text2))
    selected = {w for w, bit in zip(words, individual) if bit == 1}

    if not selected:
        return 0.0

    a = set(text1) & selected
    b = set(text2) & selected

    return len(a & b) / len(a | b) if a | b else 0.0

text1 = preprocess_text("Artificial intelligence improves education")
text2 = preprocess_text("Artificial intelligence changes technology")

words = sorted(set(text1) | set(text2))
random.seed(7)
population = initialize_population(5, len(words))

for individual in population:
    print("Individual:", individual,
          "Fitness:", round(fitness_function(individual, text1, text2), 2))


Individual: [1, 0, 1, 0, 0, 0] Fitness: 0.5
Individual: [1, 0, 0, 0, 0, 1] Fitness: 0.5
Individual: [1, 0, 0, 0, 1, 0] Fitness: 1.0
Individual: [0, 0, 0, 1, 0, 0] Fitness: 0.0
Individual: [0, 0, 1, 1, 0, 0] Fitness: 0.0


In [ ]:
import random


# 1. Selection
def selection(population, fitness_scores):
	paired = list(zip(population, fitness_scores))
	paired.sort(key=lambda item: item[1], reverse=True)

	selected = [
		individual for individual, score in paired[:2]
	]
	return selected


# 2. Crossover
def crossover(parent1, parent2):
	if len(parent1) < 2:
		return parent1[:], parent2[:]

	point = random.randint(1, len(parent1) - 1)

	child1 = parent1[:point] + parent2[point:]
	child2 = parent2[:point] + parent1[point:]

	return child1, child2


# 3. Mutation
def mutation(individual, mutation_rate=0.1):
	mutated = individual[:]

	for i in range(len(mutated)):
		if random.random() < mutation_rate:
			mutated[i] = 1 - mutated[i]

	return mutated


# Test
if __name__ == "__main__":
	population = [
		[1, 0, 1, 0],
		[1, 1, 0, 0],
		[0, 1, 1, 0],
		[0, 0, 1, 1]
	]

	fitness_scores = [0.8, 0.5, 0.9, 0.3]

	parents = selection(population, fitness_scores)

	child1, child2 = crossover(parents[0], parents[1])

	mutated_child = mutation(child1)

	print("Selected Parents:", parents)
	print("Children:", child1, child2)
	print("Mutated Child:", mutated_child)


Selected Parents: [[0, 1, 1, 0], [1, 0, 1, 0]]
Children: [0, 1, 1, 0] [1, 0, 1, 0]
Mutated Child: [0, 1, 1, 0]


In [ ]:
import random
import re
import nltk
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

#  1.Text Preprocessing
try:
    STOP_WORDS = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords")
    STOP_WORDS = set(stopwords.words("english"))

stemmer = PorterStemmer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    words = text.split()
    words = [w for w in words if w not in STOP_WORDS]
    words = [stemmer.stem(w) for w in words]

    return words


#  2. initialize_population(size, length):
    population = []
    for _ in range(size):
        individual = [random.randint(0, 1) for _ in range(length)]
        population.append(individual)
    return population

def fitness_function(individual, text1_words, text2_words):
    all_words = sorted(set(text1_words) | set(text2_words))
    selected = {w for w, bit in zip(all_words, individual) if bit == 1}

    if not selected:
        return 0.0

    common_selected = (set(text1_words) & set(text2_words)) & selected

    total_unique_words = len(set(text1_words) | set(text2_words))

    return len(common_selected) / total_unique_words if total_unique_words > 0 else 0.0


# 3. GA Operators
def selection(population, fitness_scores):
    paired = list(zip(population, fitness_scores))
    paired.sort(key=lambda item: item[1], reverse=True)
    selected = [individual for individual, score in paired[:2]]
    return selected

def crossover(parent1, parent2):
    if len(parent1) < 2:
        return parent1[:], parent2[:]

    point = random.randint(1, len(parent1) - 1)
    child1 = parent1[:point] + parent2[point:]
    child2 = parent2[:point] + parent1[point:]
    return child1, child2

def mutation(individual, mutation_rate=0.1):
    mutated = individual[:]
    for i in range(len(mutated)):
        if random.random() < mutation_rate:
            mutated[i] = 1 - mutated[i]
    return mutated


#  4. Integration & Main Execution
def run_plagiarism_checker(raw_text1, raw_text2, pop_size=10, generations=20):
    print("="*50)
    print("PLAGIARISM DETECTION SYSTEM")
    print("="*50)
    words1 = preprocess_text(raw_text1)
    words2 = preprocess_text(raw_text2)
    all_unique_words = sorted(set(words1) | set(words2))
    chrom_length = len(all_unique_words)

    if chrom_length == 0:
        print("No valid words found after preprocessing!")
        return
    population = initialize_population(pop_size, chrom_length)
    best_overall_score = 0.0
    best_overall_individual = None

    for gen in range(1, generations + 1):
        fitness_scores = [fitness_function(ind, words1, words2) for ind in population]

        max_fit = max(fitness_scores)
        if max_fit > best_overall_score:
            best_overall_score = max_fit
            best_overall_individual = population[fitness_scores.index(max_fit)]

        parents = selection(population, fitness_scores)
        next_population = [parents[0], parents[1]]

        while len(next_population) < pop_size:
            child1, child2 = crossover(parents[0], parents[1])
            next_population.append(mutation(child1))
            if len(next_population) < pop_size:
                next_population.append(mutation(child2))
        population = next_population

    # (Live Demo Output)
    similarity_percentage = best_overall_score * 100

    print(f" Text 1: {words1}")
    print(f" Text 2: {words2}\n")
    print("-" * 50)
    print(f"Best Solution Vector: {best_overall_individual}")
    print(f"Overall Similarity/Plagiarism Score: {similarity_percentage:.2f}%")

    if similarity_percentage >= 70:
        print("Status: HIGH RISK OF PLAGIARISM")
    elif similarity_percentage >= 40:
        print("Status: MODERATE SIMILARITY DETECTED")
    else:
        print("Status: LOW SIMILARITY / ORIGINAL TEXT")
    print("="*50)


# (Live Demo Example)
if __name__ == "__main__":
    original_doc = input("Enter Original Text: ")
    suspected_doc = input("Enter Suspected Text: ")

    run_plagiarism_checker(original_doc, suspected_doc)

Enter Original Text: Artificial intelligence helps students learn.
Enter Suspected Text: Artificial intelligence helps students learn faster and improves education through modern technology.
PLAGIARISM DETECTION SYSTEM
 Text 1: ['artifici', 'intellig', 'help', 'student', 'learn']
 Text 2: ['artifici', 'intellig', 'help', 'student', 'learn', 'faster', 'improv', 'educ', 'modern', 'technolog']

--------------------------------------------------
Best Solution Vector: [1, 1, 0, 1, 1, 1, 1, 0, 1, 1]
Overall Similarity/Plagiarism Score: 50.00%
Status: MODERATE SIMILARITY DETECTED
